### Basic Extraction
3/9/26

Description: 
- Explore 3 libraries, PyPDF, PDFPlumber, and pymuPDF to compare extraction capabilities

Data sources:
- https://www.usgs.gov/human-capital/nationwide-standard-position-descriptions
- https://www.commerce.gov/hr/practitioners/position-management/descriptions

In [41]:
#import needed libraries
import pandas as pd
import numpy as np
import openpyxl
import os
from openpyxl import load_workbook
import pdfplumber
import re


In [42]:
#import libraries for alternate library pypdf
import pypdf
from pypdf import PdfReader

In [43]:
#import PyMuPDF
import pymupdf

In [ ]:
#pdfplumber
# PDFplumber: extract text between "major duties" and "factor 1" section headers (headers lowercased)


def extract_major_duties_to_factor1_pdfplumber(pdf_path: str) -> str | None:
    """
    Extract the section between the header containing "major duties" and the header
    containing "factor 1". Returns the section with both headers lowercased.
    """
    with pdfplumber.open(pdf_path) as pdf:
        full_text_parts = []
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text_parts.append(text)
    full_text = "\n".join(full_text_parts)
    if not full_text.strip():
        return None

    lines = full_text.split("\n")
    start_idx = None
    end_idx = None

    for i, line in enumerate(lines):
        if "MAJOR DUTIES" in line:
            start_idx = i
            break
    if start_idx is None:
        return None

    for i in range(start_idx + 1, len(lines)):
        if "Factor 1:" in lines[i]:
            end_idx = i
            break
    if end_idx is None:
        # No "factor 1" found: take from start to end of document
        end_idx = len(lines)

    # Section: from start header (inclusive) to just before "factor 1" line
    section_lines = lines[start_idx:end_idx]

    result = "\n".join(section_lines).strip()
    return result if result else None


# Example usage:
# result = extract_major_duties_to_factor1_pdfplumber("Accountant 05 (1).pdf")
# print(result)

In [ ]:
#test pdfplumber code
txt1 = extract_major_duties_to_factor1_pdfplumber("Accountant 05 (1).pdf")
txt1

In [ ]:
#batch test code on group of PDFs of similar type

data_dict = {}  # list to hold each row dict

for i, file in enumerate(os.listdir("HR")):
    df_row = extract_major_duties_to_factor1_pdfplumber(f"HR/{file}")
    if df_row:  # Only append non-empty results
        data_dict[f"{file}"] = df_row

# #region agent log
import json
with open("debug-f1cfa6.log", "a") as _f:
    _f.write(json.dumps({"sessionId": "f1cfa6", "runId": "pre-fix", "hypothesisId": "H1", "location": "job_description_analysis_m2.ipynb:batch cell", "message": "data_dict before DataFrame", "data": {"len": len(data_dict), "value_types": [type(v).__name__ for v in (list(data_dict.values())[:3] if data_dict else [])], "sample_key": list(data_dict.keys())[0] if data_dict else None}, "timestamp": __import__("time").time() * 1000}) + "\n")
# #endregion

HR_df = pd.DataFrame([{"file_name": k, "duties": v} for k, v in data_dict.items()])
# test_df.reset_index(inplace=True, columns = ['file_name', 'duties'])  # removed: reset_index has no 'columns' param; columns already set above
#test_df.head()
#test_df.shape
display(HR_df)
HR_df.to_excel("HR_pdfplumber.xlsx")

In [ ]:
data_dict2 = {}  # list to hold each row dict

for i, file in enumerate(os.listdir("Mixed_series_PDF")):
    df_row = extract_major_duties_to_factor1_pdfplumber(f"Mixed_series_PDF/{file}")
    if df_row:  # Only append non-empty results
        data_dict2[f"{file}"] = df_row

# #region agent log
import json
with open("debug-f1cfa6.log", "a") as _f:
    _f.write(json.dumps({"sessionId": "f1cfa6", "runId": "pre-fix", "hypothesisId": "H1", "location": "job_description_analysis_m2.ipynb:batch cell", "message": "data_dict before DataFrame", "data": {"len": len(data_dict2), "value_types": [type(v).__name__ for v in (list(data_dict2.values())[:3] if data_dict2 else [])], "sample_key": list(data_dict2.keys())[0] if data_dict2 else None}, "timestamp": __import__("time").time() * 1000}) + "\n")
# #endregion

mixed_series_df = pd.DataFrame([{"file_name": k, "duties": v} for k, v in data_dict2.items()])
# test_df.reset_index(inplace=True, columns = ['file_name', 'duties'])  # removed: reset_index has no 'columns' param; columns already set above
#test_df.head()
#test_df.shape
display(mixed_series_df)
mixed_series_df.to_excel("Mixed_series_pdfplumber.xlsx")

In [ ]:
# #trying with PDF plumber shows boxes and extracts that don't make sense
# #how do we turn the boxes into columns? The quality of the extract is pretty good
# #note: explored metadata, but this did not help. Wanted to see if we can get a sense of the parameters to locate the major duties section
# with pdfplumber.open(f"HR/GS-0201-07_HRSpecialist.pdf") as pdf2:
#     first_page = pdf2.pages[1]
#     #print(first_page.chars[0])
#     first_page_text = first_page.extract_text(layout= False)
#     #explore parameters
#     first_page_words = first_page.extract_words()
#     display(first_page_words)
#     #print(pdf2.metadata)
    

In [ ]:
#check metadata
with pdfplumber.open("Accountant 05 (1).pdf") as pdf3:
    first_page = pdf3.pages[0]
    #print(first_page.chars[0])
    first_page_text = first_page.extract_text()
    print(first_page_text)
    #print(pdf3.extract_text_lines())

## Major Duties Section Extraction

Extract the "Major duties" (or "Primary responsibilities") section from position description PDFs using three libraries: **pdfplumber**, **pypdf**, and **PyMuPDF**. The section is bounded by headers like "II. MAJOR DUTIES AND RESPONSIBILITIES" and "III. FACTOR LEVELS".

In [ ]:
# pip install pymupdf  # Uncomment if not installed
import re
import fitz  # PyMuPDF

In [ ]:
# Section markers - configurable for different PD formats
START_PATTERNS = [
    # r"II\.\s*MAJOR DUTIES AND RESPONSIBILITIES",
    # r"MAJOR\s*DUTIES\s*AND\s* RESPONSIBILITIES",
    # r"Major\s*duties\s*and\s*responsibilities",
    # r"MAJOR\nDUTIES\s*AND\s*RESPONSIBILITIES",
    # r"Major\s*duties:",
    # r"Major\s*Duties:",
    # r"Primary\s*responsibilities",
    # r"MAJOR\s+DUTIES\s+AND\s+RESPONSIBILITIES"
    #4/11/26 revised this to start with Introduction
    r"Introduction",
    r"i\.\s*INTRODUCTION"
]
END_PATTERNS = [
    r"III\.\s*FACTOR LEVELS",
    r"III\.\s*FACTOR",
    r"FACTOR\s*LEVELS",
    r"Factor 1\s*[–\-]\s*Knowledge",  # "Factor 1 – Knowledge",
    r"Factor 1\s*Statements",
    r"III.\s*FACTOR\s+LEVELS and Factor\s+1\s*\S*\s*Knowledge",
    r"FACTOR\sEVALUATION\sSUMMARY"
]


def extract_major_duties_pdfplumber(pdf_path: str) -> str | None:
    """Extract Major duties section using pdfplumber."""
    full_text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text_parts.append(text)
    full_text = "\n".join(full_text_parts)
    # Clean extracted text before returning
    # Remove unreadable patterns and extraneous newlines
    cleaned_text = full_text
    # Remove line breaks
    cleaned_text = re.sub(r'\n+', ' ', cleaned_text)
    # Remove sequences of underscores with optional percentage/number and odd unicode
    cleaned_text = re.sub(r'_+\s*\d*%?\\?n?[\uf000-\uffff]*', ' ', cleaned_text)
    # Remove additional stray unicode (like \uf072 etc)
    cleaned_text = re.sub(r'[\uf000-\uffff]', ' ', cleaned_text)
    # Remove any literal "\u" unicode escape strings (not characters, but the text sequence)
    cleaned_text = re.sub(r'\\u[0-9a-fA-F]{4}', ' ', cleaned_text)
    cleaned_text = re.sub(r'\\u', ' ', cleaned_text)
    # Remove all occurrences of "/s/"
    cleaned_text = re.sub(r'/s/', ' ', cleaned_text)
    # Collapse multiple spaces
    cleaned_text = re.sub(r'\s{2,}', ' ', cleaned_text)
    full_text = cleaned_text.strip()
    return _extract_section_by_patterns(full_text)


def extract_major_duties_pypdf(pdf_path: str) -> str | None:
    """Extract Major duties section using pypdf."""
    reader = PdfReader(pdf_path)
    full_text_parts = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text_parts.append(text)
    full_text = "\n".join(full_text_parts)
    #clean output to remove unreadable text
    cleaned_text = full_text
    # Remove line breaks
    cleaned_text = re.sub(r'\n+', ' ', cleaned_text)
    # Remove sequences of underscores with optional percentage/number and odd unicode
    cleaned_text = re.sub(r'_+\s*\d*%?\\?n?[\uf000-\uffff]*', ' ', cleaned_text)
    # Remove additional stray unicode (like \uf072 etc)
    cleaned_text = re.sub(r'[\uf000-\uffff]', ' ', cleaned_text)
    # Remove any literal "\u" unicode escape strings (not characters, but the text sequence)
    cleaned_text = re.sub(r'\\u[0-9a-fA-F]{4}', ' ', cleaned_text)
    cleaned_text = re.sub(r'\\u', ' ', cleaned_text)
    # Remove all occurrences of "/s/"
    cleaned_text = re.sub(r'/s/', ' ', cleaned_text)
    # Collapse multiple spaces
    cleaned_text = re.sub(r'\s{2,}', ' ', cleaned_text)
    full_text = cleaned_text.strip()
    return _extract_section_by_patterns(full_text)


def extract_major_duties_pymupdf(pdf_path: str) -> str | None:
    """Extract Major duties section using PyMuPDF (fitz)."""
    doc = fitz.open(pdf_path)
    full_text_parts = []
    for page in doc:
        full_text_parts.append(page.get_text())
    doc.close()
    full_text = "\n".join(full_text_parts)
    #clean output to remove unreadable text
    cleaned_text = full_text
    # Remove line breaks
    cleaned_text = re.sub(r'\n+', ' ', cleaned_text)
    # Remove sequences of underscores with optional percentage/number and odd unicode
    cleaned_text = re.sub(r'_+\s*\d*%?\\?n?[\uf000-\uffff]*', ' ', cleaned_text)
    # Remove additional stray unicode (like \uf072 etc)
    cleaned_text = re.sub(r'[\uf000-\uffff]', ' ', cleaned_text)
    # Remove any literal "\u" unicode escape strings (not characters, but the text sequence)
    cleaned_text = re.sub(r'\\u[0-9a-fA-F]{4}', ' ', cleaned_text)
    cleaned_text = re.sub(r'\\u', ' ', cleaned_text)
    # Remove all occurrences of "/s/"
    cleaned_text = re.sub(r'/s/', ' ', cleaned_text)
    # Collapse multiple spaces
    cleaned_text = re.sub(r'\s{2,}', ' ', cleaned_text)
    full_text = cleaned_text.strip()
    return _extract_section_by_patterns(full_text)


def _extract_section_by_patterns(text: str) -> str | None:
    """Find text between start and end patterns (shared logic)."""
    start_match = None
    for pat in START_PATTERNS:
        start_match = re.search(pat, text, re.IGNORECASE)
        if start_match:
            #print(f"{pat}is a match")
            break
    if not start_match:
        return None
    # Start after the header line
    search_from = start_match.end()
    # Find end - look in text after start
    remainder = text[search_from:]
    end_match = None
    for pat in END_PATTERNS:
        end_match = re.search(pat, remainder, re.IGNORECASE)
        if end_match:
            break
    if end_match:
        section_text = remainder[: end_match.start()].strip()
    else:
        section_text = remainder.strip()
    return section_text if section_text else None

3/24/26 Look back to understand by the other 2 are not working
- Used Cursor code help and learned about different patterns to use to find the start and end of the section of interest

- Loosen the patterns with \s+ (or \s*)
Instead of matching a single space, match flexible whitespace:

Example idea: r"MAJOR\s+DUTIES\s+AND\s+RESPONSIBILITIES"
Same idea for III.\s*FACTOR\s+LEVELS and Factor\s+1\s*\S*\s*Knowledge if “Factor 1” lines use weird hyphen characters.

In [ ]:
# Demo: Extract Major duties from a sample PDF using all three libraries
SAMPLE_PDF = f"HR/GS-0201-05_HRSpecialist.pdf"  # or "HR/GS-0201-07_HRSpecialist.pdf"

print("=== pdfplumber ===")
result_plumber = extract_major_duties_pdfplumber(SAMPLE_PDF)
print(result_plumber[:500] + "..." if result_plumber and len(result_plumber) > 500 else result_plumber)

print("\n=== pypdf ===")
result_pypdf = extract_major_duties_pypdf(SAMPLE_PDF)
print(result_pypdf[:500] + "..." if result_pypdf and len(result_pypdf) > 500 else result_pypdf)

print("\n=== PyMuPDF ===")
result_pymupdf = extract_major_duties_pymupdf(SAMPLE_PDF)
print(result_pymupdf[:500] + "..." if result_pymupdf and len(result_pymupdf) > 500 else result_pymupdf)

In [ ]:
#batch execute using all 3 libraries and aggregate output into dataframe

data_dict = {}  # list to hold each row dict
df_test = pd.DataFrame()
file_lst = []
pdf_plumber_lst = []
pypdf_lst = []
pymupdf_lst = []

for i, file in enumerate(os.listdir("HR")):
    file_lst.append(file)
    plumb_extract = extract_major_duties_pdfplumber(f"HR/{file}")
    if plumb_extract:  # Only append non-empty results
        pdf_plumber_lst.append(plumb_extract)
    pypdf_extract = extract_major_duties_pypdf(f"HR/{file}")
    if pypdf_extract:  # Only append non-empty results
        pypdf_lst.append(pypdf_extract)
    pymupdf_extract = extract_major_duties_pymupdf(f"HR/{file}")
    if pymupdf_extract:  # Only append non-empty results
        pymupdf_lst.append(pymupdf_extract)


# # #region agent log
# import json
# with open("debug-f1cfa6.log", "a") as _f:
#     _f.write(json.dumps({"sessionId": "f1cfa6", "runId": "pre-fix", "hypothesisId": "H1", "location": "job_description_analysis_m2.ipynb:batch cell", "message": "data_dict before DataFrame", "data": {"len": len(data_dict), "value_types": [type(v).__name__ for v in (list(data_dict.values())[:3] if data_dict else [])], "sample_key": list(data_dict.keys())[0] if data_dict else None}, "timestamp": __import__("time").time() * 1000}) + "\n")
# # #endregion

HR_df = pd.DataFrame({
    'file_name': file_lst,
    'pdfplumber_output': pdf_plumber_lst,
    'pypdf_output': pypdf_lst,
    'pymupdf_output': pymupdf_lst 
    })
# test_df.reset_index(inplace=True, columns = ['file_name', 'duties'])  # removed: reset_index has no 'columns' param; columns already set above
#test_df.head()
#test_df.shape
display(HR_df)

#remove illegal formats that prevent output from showing from pymupdf
HR_df= HR_df.map(lambda x: x.encode('unicode_escape').decode('utf-8') if isinstance(x, str) else x)

HR_df.to_excel("Multi_library_df_test8.xlsx", index = False)


## 4/13 Text processing

In [ ]:
#pip install nltk

In [ ]:
#pip install sklearn

In [44]:
#import packages

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import string
import nltk

In [ ]:
#from sklearn.metrics import precision_score, recall_score, f1_score

In [46]:
# Download required NLTK resources; googled and downloading "punkt_tab" worked over "punkt"
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\st212\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\st212\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
print(nltk.data.path)

In [ ]:
def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Tokenize
    tokens = word_tokenize(text)
    # 3. Remove punctuation and stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    # 4. Stemming (optional but common)
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(word) for word in tokens]
    #4/15/26 - set type raised errors when trying to calculate precision,recall, f1
    return set(stemmed_tokens) # Using a set for set-based evaluation

# Example usage
raw_text = "NLTK is a great tool for Natural Language Processing!"
processed = preprocess_text(raw_text)
print(processed)

tokens = word_tokenize(raw_text)
print(tokens)
set(tokens)

{'nltk', 'process', 'languag', 'tool', 'great', 'natur'}
['NLTK', 'is', 'a', 'great', 'tool', 'for', 'Natural', 'Language', 'Processing', '!']


{'!',
 'Language',
 'NLTK',
 'Natural',
 'Processing',
 'a',
 'for',
 'great',
 'is',
 'tool'}

In [48]:
#import "Multi_library_df_test8.xlsx" as HR_df2 with ground truth

HR_df2 = pd.read_excel("Multi_library_df_test8.xlsx")
HR_df2.head()

,file_name,pdfplumber_output,pypdf_output,pymupdf_output,ground_truth
0,GS-0150-05-GEOGRAPHER_508.pdf,The incumbent serves as a trainee assigned to ...,The incumbent serves as a trainee assigned to ...,The incumbent serves as a trainee assigned to ...,The incumbent serves as a trainee assigned to ...
1,GS-0201-05_HRSpecialist.pdf,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...
2,GS-0201-07_HRSpecialist.pdf,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...
3,GS-0201-09_HRSpecialist.pdf,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...
4,GS-0201-11_HRSpecialist.pdf,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...,This position is located in a Servicing Human ...


In [53]:
#apply nltk processing to df using df.map; apply only to text columns, not file_name

text_cols = ['pdfplumber_output', 'pypdf_output', 'pymupdf_output', 'ground_truth']
token_df = HR_df2[text_cols].map(preprocess_text)

token_df.head()

,pdfplumber_output,pypdf_output,pymupdf_output,ground_truth
0,"{effect, similar, indic, process, scientif, an...","{effect, similar, indic, process, scientif, an...","{effect, similar, indic, process, scientif, an...","{effect, similar, indic, process, scientif, an..."
1,"{process, direct, cba, union, basic, make, tim...","{process, direct, cba, union, basic, make, tim...","{process, direct, cba, union, basic, make, tim...","{process, direct, cba, union, basic, make, tim..."
2,"{moder, owcp, process, union, basic, result, c...","{moder, owcp, process, union, basic, result, c...","{moder, owcp, process, union, basic, result, c...","{moder, owcp, process, union, basic, result, c..."
3,"{moder, owcp, disput, process, direct, union, ...","{moder, owcp, disput, process, direct, union, ...","{moder, owcp, disput, process, direct, union, ...","{moder, owcp, disput, process, direct, union, ..."
4,"{owcp, disput, failur, process, direct, unprec...","{owcp, disput, failur, process, direct, unprec...","{owcp, disput, failur, process, direct, unprec...","{owcp, disput, failur, process, direct, unprec..."


In [70]:
def prf1_sets(pred, ref):
    pred, ref = pred or set(), ref or set()
    inter = len(pred & ref)
    p = inter / len(pred) if pred else (1.0 if not ref else 0.0)
    r = inter / len(ref) if ref else (1.0 if not pred else 0.0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1

In [67]:
type(token_df.loc[0,"pdfplumber_output"])

set

In [68]:
pred = token_df.loc[0,"pdfplumber_output"]
ref = token_df.loc[0,"ground_truth"]

prf1_sets(pred, ref)

(0.952054794520548, 0.9025974025974026)

In [78]:
for col in ['pdfplumber_output', 'pypdf_output', 'pymupdf_output']:
    ps, rs, fs = [], [], []
    for i, row in token_df.iterrows():
        p, r, f1 = prf1_sets(token_df.loc[i, col], token_df.loc[i,'ground_truth'])
        ps.append(p); rs.append(r); fs.append(f1)
    print(col, f"The precision is {round(np.mean(ps),3)}.", f"The recall is {round(np.mean(rs),3)}.", f"The f1score is {round(np.mean(fs),3)}.")

pdfplumber_output The precision is 0.925. The recall is 0.985. The f1score is 0.952.
pypdf_output The precision is 0.925. The recall is 0.985. The f1score is 0.952.
pymupdf_output The precision is 0.925. The recall is 0.985. The f1score is 0.952.


In [ ]:
#calculate f1, precision, and recall
#Tried this method but was not able to get this to work

from nltk.metrics.scores import precision, recall, f_measure

# Example evaluation sets (sets of preprocessed tokens)
reference = (token_df['pdfplumber_output'])
test = (token_df['ground_truth'])

# True Positives (TP): Intersection of reference and test
tp = reference.intersection(test) # {'nltk', 'great', 'tool'}
# False Positives (FP): In test, not in reference
fp = test.difference(reference) # {'bad', 'system'}
# False Negatives (FN): In reference, not in test
fn = reference.difference(test) # {'nlp', 'process'}

# Calculate metrics
p = precision(reference, test) # TP / (TP + FP)
r = recall(reference, test)    # TP / (TP + FN)
f1 = f_measure(reference, test) # 2 * (P * R) / (P + R)

print(f"Precision: {p:.2f}")
print(f"Recall: {r:.2f}")
print(f"F1-Score: {f1:.2f}")

## Additional pymupdf exploration

In [ ]:
HR_df['pymupdf_output']

In [ ]:


HR_df= HR_df.map(lambda x: x.encode('unicode_escape').decode('utf-8') if isinstance(x, str) else x)

HR_df['pymupdf_output'].to_excel("test.xlsx")

In [ ]:
#batch execute using each library - pdfplumber

data_dict = {}  # list to hold each row dict

for i, file in enumerate(os.listdir("HR")):
    df_row = extract_major_duties_pdfplumber(f"HR/{file}")
    if df_row:  # Only append non-empty results
        data_dict[f"{file}"] = df_row

# # #region agent log
# import json
# with open("debug-f1cfa6.log", "a") as _f:
#     _f.write(json.dumps({"sessionId": "f1cfa6", "runId": "pre-fix", "hypothesisId": "H1", "location": "job_description_analysis_m2.ipynb:batch cell", "message": "data_dict before DataFrame", "data": {"len": len(data_dict), "value_types": [type(v).__name__ for v in (list(data_dict.values())[:3] if data_dict else [])], "sample_key": list(data_dict.keys())[0] if data_dict else None}, "timestamp": __import__("time").time() * 1000}) + "\n")
# # #endregion

HR_df = pd.DataFrame([{"file_name": k, "duties": v} for k, v in data_dict.items()])
# test_df.reset_index(inplace=True, columns = ['file_name', 'duties'])  # removed: reset_index has no 'columns' param; columns already set above
#test_df.head()
#test_df.shape
display(HR_df)
HR_df.to_excel("HR_pdfplumber_test.xlsx")

In [ ]:
#test with pypdf
data_dict = {}  # list to hold each row dict

for i, file in enumerate(os.listdir("HR")):
    df_row = extract_major_duties_pypdf(f"HR/{file}")
    if df_row:  # Only append non-empty results
        data_dict[f"{file}"] = df_row


HR_df = pd.DataFrame([{"file_name": k, "duties": v} for k, v in data_dict.items()])
# test_df.reset_index(inplace=True, columns = ['file_name', 'duties'])  # removed: reset_index has no 'columns' param; columns already set above
#test_df.head()
#test_df.shape
display(HR_df)
HR_df.to_excel("HR_pypdf_test.xlsx")

In [ ]:
#test with pymupdf
data_dict = {}  # list to hold each row dict

for i, file in enumerate(os.listdir("HR")):
    df_row = extract_major_duties_pymupdf(f"HR/{file}")
    if df_row:  # Only append non-empty results
        data_dict[f"{file}"] = df_row

# # #region agent log
# import json
# with open("debug-f1cfa6.log", "a") as _f:
#     _f.write(json.dumps({"sessionId": "f1cfa6", "runId": "pre-fix", "hypothesisId": "H1", "location": "job_description_analysis_m2.ipynb:batch cell", "message": "data_dict before DataFrame", "data": {"len": len(data_dict), "value_types": [type(v).__name__ for v in (list(data_dict.values())[:3] if data_dict else [])], "sample_key": list(data_dict.keys())[0] if data_dict else None}, "timestamp": __import__("time").time() * 1000}) + "\n")
# # #endregion

HR_df = pd.DataFrame([{"file_name": k, "duties": v} for k, v in data_dict.items()])
# test_df.reset_index(inplace=True, columns = ['file_name', 'duties'])  # removed: reset_index has no 'columns' param; columns already set above
#test_df.head()
#test_df.shape
display(HR_df)
HR_df.to_excel("HR_pymupdf_test.xlsx")

## pypdf

In [ ]:
#pip install pypdf

In [ ]:
#3/3/26 used dummy code from google AI search
#extracts data on cover page only

def extract_form_data(pdf_path):
    reader = PdfReader(pdf_path)
    form_data = {}
    
    # Iterate over pages and get fields
    for page in reader.pages[0]:
        # get_fields() returns a dictionary of field data
        fields = reader.get_fields()
        if fields:
            for field_name, field_data in fields.items():
                # Extract the value (/V) from the field data
                form_data[field_name] = field_data.get('/V', '')
                
    return form_data



In [ ]:
#display
pd.set_option('display.max_rows', None, 'display.max_columns', None)

#execute function
test_dict = extract_form_data(f"HR/GS-0201-07_HRSpecialist.pdf")
test_df = pd.DataFrame(test_dict, index=[0])
test_df.head()

In [ ]:
for file in os.listdir("HR"):
    print(file)

In [ ]:
#test running on a several PDs

# Improved approach: build up a list of dicts, then build DataFrame once at end,
# and make sure the dataframe is being expanded, instead of unused 'pdf_df'

rows = []  # list to hold each row dict

for i, file in enumerate(os.listdir("HR")):
    df_row = extract_form_data(f"HR/{file}")
    if df_row:  # Only append non-empty results
        rows.append(df_row)

test_df = pd.DataFrame(rows)
test_df.reset_index(drop=True, inplace=True)
#test_df.head()
#test_df.shape
display(test_df)



In [ ]:
#export to excel
test_df.to_excel("test_df.xlsx", index=False)